### Transformação de dados na Camada Silver

Leitura dos dados da camada Bronze, aplicação das regras de qualidade e integridade referencial, e armazenamento dos dados tratados na camada Silver.

In [0]:
# CARREGAMENTO DE VARIÁVEIS DE AMBIENTE
import os
from dotenv import load_dotenv
from pyspark.sql.functions import col, current_timestamp, year, month, trim, when, lit, row_number, to_date, to_timestamp,coalesce
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType, TimestampType
from delta.tables import DeltaTable

load_dotenv()

client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")

# DICIONÁRIO DE CREDENCIAIS OAUTH PARA USAR COM SPARK
adls_options = {
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# DICIONÁRIO DE PATHS
paths = {
    "food_avaliacoes_produto_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_avaliacoes_produto",
    "food_estoque_lojas_bronze": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/bronze/food_estoque_lojas",
    "food_avaliacoes_produto_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_avaliacoes_produto",
    "food_estoque_lojas_silver": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver/food_estoque_lojas",
    "raw_path": f"abfss://raw@{storage_account_name}.dfs.core.windows.net/batch-data",
    "silver_path": f"abfss://squad3@{storage_account_name}.dfs.core.windows.net/silver"
}

In [0]:
# LEITURA DAS TABELAS BRONZE
df_bronze_avaliacoes = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_avaliacoes_produto_bronze"])
)

df_bronze_estoque = (
    spark.read.format("delta")
    .options(**adls_options)
    .load(paths["food_estoque_lojas_bronze"])
)

# LEITURA DAS TABELAS DE REFERÊNCIA VINDAS DA CAMADA SILVER
# SQUAD 3
df_physical_lojas = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_lojas.csv")
)

df_physical_produtos_pereciveis = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/physical_produtos_pereciveis.csv")
)

df_food_lotes_producao = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/food_lotes_producao.csv")
)

# SQUAD 1
df_ecommerce_produtos = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_produtos.csv")
)

df_ecommerce_pedidos = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{paths['raw_path']}/ecommerce_pedidos.csv")
)


In [0]:
# SILVER - food_estoque_lojas Dataframe - trim de strings e cast explícito
df_silver_estoque = (
    df_bronze_estoque
    .withColumn("id_loja", col("id_loja").cast("integer"))
    .withColumn("sku", trim(col("sku")))
    .withColumn("id_lote", col("id_lote").cast("integer"))
    .withColumn("quantidade_disponivel", col("quantidade_disponivel").cast("integer"))
    .withColumn("estoque_minimo", col("estoque_minimo").cast("integer"))
    .withColumn("dt_snapshot", col("dt_snapshot").cast("date"))
)


# REGRA 1 - id_loja deve existir em physical_lojas
df_silver_estoque = (
    df_silver_estoque.alias("e")
    .join(
        df_physical_lojas
        .select(col("id_loja").alias("id_loja_ref"))
        .alias("l"),
        col("e.id_loja") == col("l.id_loja_ref"),
        "left"
    )
    .withColumn(
        "flag_id_loja_invalido",
        when(col("l.id_loja_ref").isNull(), lit(True))
        .otherwise(lit(False))
    )
    .drop("id_loja_ref")
)


# REGRA 2 - Sku deve existir no catálogo unificado(ecommerce_produtos e physical_produtos_pereciveis)
df_skus_ecommerce = (
    df_ecommerce_produtos
    .select(trim(col("sku")).alias("sku"))
)
df_skus_pereciveis = (
    df_physical_produtos_pereciveis
    .select(trim(col("sku")).alias("sku"))
)
df_catalogo_unificado = (
    df_skus_ecommerce
    .unionByName(df_skus_pereciveis)
    .dropDuplicates(["sku"])
)
print(f"SKUs ecommerce: {df_skus_ecommerce.count()}, SKUs perecíveis: {df_skus_pereciveis.count()}, SKUs catálogo unificado: {df_catalogo_unificado.count()}")

df_silver_estoque = (
    df_silver_estoque.alias("e")
    .join(
        df_catalogo_unificado
        .select(col("sku").alias("sku_catalogo"))
        .alias("c"),
        col("e.sku") == col("c.sku_catalogo"),
        "left"
    )
    .withColumn(
        "flag_sku_invalido",
        when(col("c.sku_catalogo").isNull(), lit(True))
        .otherwise(lit(False))
    )
    .drop("sku_catalogo")
)


# REGRA 3 - Quantidade_disponivel deve ser >= 0
df_silver_estoque = df_silver_estoque.withColumn(
    "flag_quantidade_invalida",
    when(
        col("quantidade_disponivel").isNull() |
        (col("quantidade_disponivel") < 0),
        lit(True)
    ).otherwise(lit(False))
)


# REGRAS 4 e 5 - Validação do id_lote e janela de validade
df_silver_estoque = (
    df_silver_estoque.alias("e")
    .join(
        df_food_lotes_producao
        .select(
            "id_lote",
            "dt_fabricacao",
            "dt_validade"
        )
        .alias("l"),
        col("e.id_lote") == col("l.id_lote"),
        "left"
    )
    .withColumn(
        "flag_id_lote_invalido",
        when(
            # REGRA 4: id_lote preenchido no estoque, mas não encontrado em food_lotes_producao
            col("e.id_lote").isNotNull()
            & col("l.id_lote").isNull(),
            lit(True)
        )
        .otherwise(lit(False))
    )
    .withColumn(
        "flag_dt_snapshot_invalido",
        when(
            # REGRA 5: dt_snapshot fora do intervalo [dt_fabricacao, dt_validade]
            col("e.id_lote").isNotNull()
            & col("l.id_lote").isNotNull()
            & (
                (to_timestamp(col("e.dt_snapshot")) < to_timestamp(col("l.dt_fabricacao")))
                | (to_timestamp(col("e.dt_snapshot")) > to_timestamp(col("l.dt_validade")))
            ),
            lit(True)
        )
        .otherwise(lit(False))
    )
    .drop(col("l.id_lote"),col("l.dt_fabricacao"),col("l.dt_validade"))
)

# STATUS GERAL DE QUALIDADE
df_silver_estoque = (
    df_silver_estoque
    .withColumn(
        "quality_issue",
        (
            col("flag_id_loja_invalido") |
            col("flag_sku_invalido") |
            col("flag_quantidade_invalida") |
            col("flag_id_lote_invalido") |
            col("flag_dt_snapshot_invalido")
        )
    )
    .withColumn("quality_status",
                when(col("quality_issue"),lit("REVIEW")).otherwise(lit("OK")))
)


In [0]:
# SILVER - food_avaliacoes_produto Dataframe - trim de strings e cast explícito
df_silver_avaliacoes = (
    df_bronze_avaliacoes
    .withColumn("id_avaliacao", col("id_avaliacao").cast("integer"))
    .withColumn("id_pedido", col("id_pedido").cast("integer"))
    .withColumn("id_cliente", col("id_cliente").cast("integer"))
    .withColumn("sku", trim(col("sku")))
    .withColumn("nota", col("nota").cast("integer"))
    .withColumn("dt_avaliacao", col("dt_avaliacao").cast("timestamp"))
    .withColumn("verificada", col("verificada").cast("boolean"))
)

# REGRA 1
# REMOÇÃO DE id_avaliacao NULO
total_id_old = df_silver_avaliacoes.count()

df_silver_avaliacoes = (
    df_silver_avaliacoes
    .filter(col("id_avaliacao").isNotNull())
)

total_id_final = df_silver_avaliacoes.count()

print(f"Total antes: {total_id_old}")
print(f"Total após remoção de id_avaliacao nulo: {total_id_final}")
print(f"IDs nulos removidos: {total_id_old - total_id_final}")

# REMOÇÃO DE DUPLICIDADES
window_pk = (
    Window
    .partitionBy("id_avaliacao")
    .orderBy(col("bronze_ingested_at").desc(),col("bronze_source_file").desc()))

total_antes_duplicidade = df_silver_avaliacoes.count()

df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn("_rn", row_number().over(window_pk))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
total_depois_duplicidade = df_silver_avaliacoes.count()
print(f"Duplicidades removidas: {total_antes_duplicidade - total_depois_duplicidade}")


# REGRA 2
# nota deve ser um inteiro entre 1 e 5
df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "flag_nota_invalida",
        when(col("nota").isNull() |(col("nota") < 1) |(col("nota") > 5),lit(True)).otherwise(lit(False)))
)


# REGRA 3 - Quando verificada = TRUE, id_pedido não pode ser nulo e deve existir em ecommerce_pedidos
df_silver_avaliacoes = (
    df_silver_avaliacoes.alias("a")
    .join(
        df_ecommerce_pedidos
        .select("id_pedido")
        .alias("p"),
        col("a.id_pedido") == col("p.id_pedido"),
        "left"
    )
    .withColumn(
        "flag_avaliacao_pedido_verificado_invalido",
        when(
            (col("a.verificada") == True)
            & (
                col("a.id_pedido").isNull()
                | col("p.id_pedido").isNull()
            ),
            lit(True)
        )
        .otherwise(lit(False))
    )
    .drop(col("p.id_pedido"))
)


# REGRA 4 - Quando verificada = FALSE, id_pedido deve ser nulo
df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "flag_avaliacao_pedido_nao_verificado_invalido",
        when(
            col("verificada") == False,
            when(col("id_pedido").isNotNull(),lit(True)).otherwise(lit(False))
        ).otherwise(lit(False))
    )
)


# REGRA 5 - SKU deve existir no catálogo unificado
df_silver_avaliacoes = (
    df_silver_avaliacoes.alias("a")
    .join(
        df_catalogo_unificado
        .select("sku")
        .alias("c"),
        col("a.sku") == col("c.sku"),
        "left"
    )
    .withColumn(
        "flag_sku_invalido",
        when(
            col("a.sku").isNull() |
            col("c.sku").isNull(),
            lit(True)
        )
        .otherwise(lit(False))
    )
    .drop(col("c.sku"))
)


# STATUS GERAL DE QUALIDADE
df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "quality_issue",
        (
            col("flag_nota_invalida") |
            col("flag_avaliacao_pedido_verificado_invalido") |
            col("flag_avaliacao_pedido_nao_verificado_invalido") |
            col("flag_sku_invalido")
        )
    )
    .withColumn(
        "quality_status",
        when(col("quality_issue"),lit("REVIEW")).otherwise(lit("OK"))
    )
)


In [0]:
# COLUNAS DE AUDITORIA
df_silver_estoque = (
    df_silver_estoque
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)

df_silver_avaliacoes = (
    df_silver_avaliacoes
    .withColumn(
        "silver_processed_at",
        current_timestamp()
    )
)

In [0]:
# GRAVAÇÃO SILVER - FOOD AVALIAÇÕES PRODUTO

# Upsert lógico via LEFT ANTI + UNION
try:
    df_avaliacoes_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["food_avaliacoes_produto_silver"])
    )

    # Confirma que a leitura realmente conseguiu acessar a Silver
    df_avaliacoes_atual.limit(1).collect()

    silver_avaliacoes_existe = True

except Exception:
    silver_avaliacoes_existe = False


if not silver_avaliacoes_existe:
    # Primeira carga
    df_avaliacoes_final = df_silver_avaliacoes

    print("Silver de food_avaliacoes_produto não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:
    # Chaves que chegaram nesta carga
    chaves_avaliacoes = (
        df_silver_avaliacoes
        .select("id_avaliacao")
        .dropDuplicates()
    )
    # Mantém somente registros antigos que NÃO chegaram novamente
    df_avaliacoes_mantidas = (
        df_avaliacoes_atual
        .join(
            chaves_avaliacoes,
            on="id_avaliacao",
            how="left_anti"
        )
    )
    # Junta:
    # - registros novos/atualizados
    # - registros antigos que não foram afetados
    df_avaliacoes_final = (
        df_silver_avaliacoes
        .unionByName(df_avaliacoes_mantidas)
    )
    print("Silver de food_avaliacoes_produto já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


# Grava a Silver
(
    df_avaliacoes_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .partitionBy("ano", "mes")
    .save(paths["food_avaliacoes_produto_silver"])
)
print("✅ Silver de food_avaliacoes_produto gravada com sucesso.")


# GRAVAÇÃO SILVER - FOOD ESTOQUE LOJAS
# Upsert lógico via LEFT ANTI + UNION
try:
    df_estoque_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(paths["food_estoque_lojas_silver"])
    )

    # Confirma que a leitura realmente conseguiu acessar a Silver
    df_estoque_atual.limit(1).collect()

    silver_estoque_existe = True

except Exception:
    silver_estoque_existe = False


if not silver_estoque_existe:
    # Primeira carga
    df_estoque_final = df_silver_estoque
    print("Silver de food_estoque_lojas não existe.")
    print("Primeira carga: todos os registros serão gravados.")

else:
    # Chaves que chegaram nesta carga
    chaves_estoque = (
        df_silver_estoque
        .select(
            "id_loja",
            "sku",
            "dt_snapshot"
        )
        .dropDuplicates()
    )
    # Mantém somente registros antigos que NÃO chegaram novamente
    df_estoque_mantidas = (
        df_estoque_atual
        .join(
            chaves_estoque,
            on=[
                "id_loja",
                "sku",
                "dt_snapshot"
            ],
            how="left_anti"
        )
    )
    # Junta:
    # - registros novos/atualizados
    # - registros antigos que não foram afetados
    df_estoque_final = (
        df_silver_estoque
        .unionByName(df_estoque_mantidas)
    )

    print("Silver de food_estoque_lojas já existe.")
    print("Upsert lógico realizado via LEFT ANTI + UNION.")


# Grava a Silver
(
    df_estoque_final
    .write
    .format("delta")
    .options(**adls_options)
    .mode("overwrite")
    .option("overwriteSchema", "false")
    .partitionBy("ano", "mes")
    .save(paths["food_estoque_lojas_silver"])
)

print("✅ Silver de food_estoque_lojas gravada com sucesso.")

In [0]:
# LEITURA FINAL DA SILVER
df_final_avaliacoes = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(paths["food_avaliacoes_produto_silver"])
)
display(df_final_avaliacoes.limit(5))

df_final_estoque = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(paths["food_estoque_lojas_silver"])
)
display(df_final_estoque.limit(5))

# VALIDAÇÃO FINAL
print(f"food_avaliacoes_produto Silver: {df_final_avaliacoes.count()} registros")
print(f"food_estoque_lojas Silver: {df_final_estoque.count()} registros")

# DISTRIBUIÇÃO DA QUALIDADE
print("FOOD AVALIACOES PRODUTO")
df_final_avaliacoes.groupBy("quality_status").count().show()
print("FOOD ESTOQUE LOJAS")
df_final_estoque.groupBy("quality_status").count().show()
      